In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/TESS Toronto emotional speech set data"

print(os.listdir(dataset_path))


In [ ]:
sample_file = "/content/drive/MyDrive/TESS Toronto emotional speech set data/OAF_angry/OAF_germ_angry.wav"

In [ ]:
import librosa

audio, sr = librosa.load(sample_file, sr=16000)

print(sr)
print(len(audio))

In [ ]:
import librosa
import numpy as np

mfcc = librosa.feature.mfcc(
    y=audio,
    sr=sr,
    n_mfcc=40
)

print("MFCC Shape:", mfcc.shape)

In [ ]:
import librosa.display
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))

librosa.display.specshow(
    mfcc,
    x_axis='time'
)

plt.colorbar()

plt.title("MFCC Features")

plt.show()

In [ ]:
mfcc_mean = np.mean(mfcc.T, axis=0)

print(mfcc_mean.shape)

In [ ]:
emotion = sample_file.split("/")[-2]

print(emotion)

In [ ]:
import os
import librosa
import numpy as np

In [ ]:
dataset_path = "/content/drive/MyDrive/TESS Toronto emotional speech set data"

In [ ]:
X = []
y = []

In [ ]:
for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):

        print("Processing:", folder)

        for file in os.listdir(folder_path):

            file_path = os.path.join(folder_path, file)

            try:

                # LOAD AUDIO
                audio, sr = librosa.load(
                    file_path,
                    sr=16000
                )

                # EXTRACT MFCC
                mfcc = librosa.feature.mfcc(
                    y=audio,
                    sr=sr,
                    n_mfcc=40
                )

                # CONVERT TO FIXED SIZE
                mfcc_mean = np.mean(
                    mfcc.T,
                    axis=0
                )

                # SAVE FEATURES
                X.append(mfcc_mean)

                # SAVE LABEL
                y.append(folder)

            except Exception as e:

                print("ERROR:", file_path)
                print(e)

Processing: YAF_neutral
Processing: YAF_sad
Processing: YAF_happy
Processing: YAF_fear
Processing: YAF_disgust
Processing: YAF_pleasant_surprised
Processing: OAF_Sad


In [ ]:
X = np.array(X)
y = np.array(y)

In [ ]:
print("Feature Shape:", X.shape)
print("Label Shape:", y.shape)

In [ ]:
print(y[:10])

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

In [ ]:
print(y[:5])

print(y_encoded[:5])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

In [ ]:
print(X_train_tensor.shape)
print(y_train_tensor.shape)


In [ ]:
class EmotionClassifier(nn.Module):

    def __init__(self):

        super(EmotionClassifier, self).__init__()

        self.fc1 = nn.Linear(40, 128)

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(128, 64)

        self.fc3 = nn.Linear(
            64,
            len(np.unique(y_encoded))
        )

    def forward(self, x):

        x = self.fc1(x)

        x = self.relu(x)

        x = self.fc2(x)

        x = self.relu(x)

        x = self.fc3(x)

        return x

In [ ]:
model = EmotionClassifier()

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs = 100

for epoch in range(epochs):

    # FORWARD PASS
    outputs = model(X_train_tensor)

    # CALCULATE LOSS
    loss = criterion(
        outputs,
        y_train_tensor
    )

    # RESET GRADIENTS
    optimizer.zero_grad()

    # BACKPROPAGATION
    loss.backward()

    # UPDATE WEIGHTS
    optimizer.step()

    # PRINT LOSS
    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )

In [ ]:
with torch.no_grad():

    outputs = model(X_test_tensor)

    _, predicted = torch.max(outputs, 1)

    accuracy = (
        predicted == y_test_tensor
    ).sum().item() / len(y_test_tensor)

print(f"Accuracy: {accuracy*100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
cm = confusion_matrix(
    y_test_tensor,
    predicted
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:
np.mean(mfcc.T, axis=0)

In [ ]:
MAX_PAD_LEN = 200

X = []
y = []

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):

        print("Processing:", folder)

        for file in os.listdir(folder_path):

            file_path = os.path.join(folder_path, file)

            try:

                # LOAD AUDIO
                audio, sr = librosa.load(
                    file_path,
                    sr=16000
                )

                # EXTRACT MFCC
                mfcc = librosa.feature.mfcc(
                    y=audio,
                    sr=sr,
                    n_mfcc=40
                )

                # PAD OR TRUNCATE
                if mfcc.shape[1] < MAX_PAD_LEN:

                    pad_width = MAX_PAD_LEN - mfcc.shape[1]

                    mfcc = np.pad(
                        mfcc,
                        pad_width=((0,0),(0,pad_width)),
                        mode='constant'
                    )

                else:

                    mfcc = mfcc[:, :MAX_PAD_LEN]

                X.append(mfcc)

                y.append(folder)

            except Exception as e:

                print("ERROR:", file_path)
                print(e)

In [ ]:
X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [ ]:
import torch

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

In [ ]:
X_train_tensor = X_train_tensor.unsqueeze(1)

X_test_tensor = X_test_tensor.unsqueeze(1)

print(X_train_tensor.shape)

In [ ]:
import torch.nn as nn

class CNNLSTM(nn.Module):

    def __init__(self, num_classes):

        super(CNNLSTM, self).__init__()

        # CNN PART
        self.cnn = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        # LSTM PART
        self.lstm = nn.LSTM(
            input_size=32 * 20,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        # CLASSIFIER
        self.fc = nn.Linear(
            64,
            num_classes
        )

    def forward(self, x):

        # CNN
        x = self.cnn(x)

        # SHAPE:
        # batch, channel, height, width

        batch_size, channels, height, width = x.size()

        # RESHAPE FOR LSTM
        x = x.permute(0, 3, 1, 2)

        x = x.contiguous().view(
            batch_size,
            width,
            channels * height
        )

        # LSTM
        lstm_out, _ = self.lstm(x)

        # LAST TIME STEP
        x = lstm_out[:, -1, :]

        # CLASSIFIER
        x = self.fc(x)

        return x

In [ ]:
num_classes = len(np.unique(y_encoded))

model = CNNLSTM(num_classes)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
print(type(X))
print(type(y))

print(len(X))
print(len(y))

In [ ]:
print(X[0].shape)
print(X[1].shape)
print(X[2].shape)

In [ ]:
X = np.array(X, dtype=np.float32)

print(X.shape)
print(X.dtype)

In [ ]:
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [ ]:
import torch

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

print(X_train_tensor.shape)

In [ ]:
X_train_tensor = X_train_tensor.unsqueeze(1)

X_test_tensor = X_test_tensor.unsqueeze(1)

print(X_train_tensor.shape)

In [ ]:
import torch
import torch.nn as nn

class CNNLSTM(nn.Module):

    def __init__(self, num_classes):

        super(CNNLSTM, self).__init__()

        # CNN BLOCK
        self.cnn = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        # LSTM BLOCK
        self.lstm = nn.LSTM(
            input_size=32 * 20,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        # CLASSIFIER
        self.fc = nn.Linear(
            64,
            num_classes
        )

    def forward(self, x):

        # CNN
        x = self.cnn(x)

        # SHAPE
        batch_size, channels, height, width = x.size()
        print(x.shape)

        # RESHAPE FOR LSTM
        x = x.permute(0, 3, 1, 2)

        x = x.contiguous().view(
            batch_size,
            width,
            channels * height
        )

        # LSTM
        lstm_out, _ = self.lstm(x)

        # LAST OUTPUT
        x = lstm_out[:, -1, :]

        # FINAL CLASSIFIER
        x = self.fc(x)

        return x

In [ ]:
num_classes = len(np.unique(y_encoded))

model = CNNLSTM(num_classes)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
print(X_train_tensor.shape)

In [ ]:
X_train_tensor = X_train_tensor.unsqueeze(1)

X_test_tensor = X_test_tensor.unsqueeze(1)

In [ ]:
print(X_train_tensor.shape)

In [ ]:
print("X_train_tensor shape:")
print(X_train_tensor.shape)

print("\nOne sample shape:")
print(X_train_tensor[0].shape)

print("\nModel test:")
test_input = X_train_tensor[:2]

print(test_input.shape)

In [ ]:
MAX_PAD_LEN = 200

X = []
y = []

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):

        print("Processing:", folder)

        for file in os.listdir(folder_path):

            file_path = os.path.join(folder_path, file)

            try:

                # LOAD AUDIO
                audio, sr = librosa.load(
                    file_path,
                    sr=16000
                )

                # EXTRACT MFCC
                mfcc = librosa.feature.mfcc(
                    y=audio,
                    sr=sr,
                    n_mfcc=40
                )

                # IMPORTANT:
                # DO NOT USE np.mean()

                # PAD / TRUNCATE
                if mfcc.shape[1] < MAX_PAD_LEN:

                    pad_width = MAX_PAD_LEN - mfcc.shape[1]

                    mfcc = np.pad(
                        mfcc,
                        pad_width=((0,0),(0,pad_width)),
                        mode='constant'
                    )

                else:

                    mfcc = mfcc[:, :MAX_PAD_LEN]

                X.append(mfcc)

                y.append(folder)

            except Exception as e:

                print("ERROR:", file_path)
                print(e)

In [ ]:
X = np.array(X, dtype=np.float32)

y = np.array(y)

print(X.shape)

In [ ]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

In [ ]:
X_train_tensor = X_train_tensor.unsqueeze(1)

X_test_tensor = X_test_tensor.unsqueeze(1)

print(X_train_tensor.shape)

In [ ]:
test_output = model(X_train_tensor[:2])

print(test_output.shape)

In [ ]:
num_classes = len(np.unique(y_encoded))

model = CNNLSTM(num_classes)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs = 30

for epoch in range(epochs):

    # FORWARD PASS
    outputs = model(X_train_tensor)

    # LOSS
    loss = criterion(
        outputs,
        y_train_tensor
    )

    # RESET GRADIENTS
    optimizer.zero_grad()

    # BACKPROPAGATION
    loss.backward()

    # UPDATE WEIGHTS
    optimizer.step()

    # PRINT LOSS
    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {loss.item():.4f}"
    )

In [ ]:
with torch.no_grad():

    outputs = model(X_test_tensor)

    _, predicted = torch.max(outputs, 1)

    accuracy = (
        predicted == y_test_tensor
    ).sum().item() / len(y_test_tensor)

print(f"Accuracy: {accuracy*100:.2f}%")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
num_classes = len(np.unique(y_encoded))

model = CNNLSTM(num_classes)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs, labels in train_loader:

        # FORWARD
        outputs = model(inputs)

        loss = criterion(outputs, labels)

        # BACKWARD
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {avg_loss:.4f}"
    )

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in test_loader:

        outputs = model(inputs)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):

    def __init__(self, num_classes):

        super(SimpleCNN, self).__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.flatten = nn.Flatten()

        self.fc = nn.Sequential(

            nn.Linear(
                64 * 10 * 50,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, x):

        x = self.cnn(x)

        x = self.flatten(x)

        x = self.fc(x)

        return x

In [ ]:
num_classes = len(np.unique(y_encoded))

model = SimpleCNN(num_classes)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs, labels in train_loader:

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {avg_loss:.4f}"
    )

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in test_loader:

        outputs = model(inputs)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
torch.save(
    model.state_dict(),
    "speech_emotion_model.pth"
)

print("Model Saved")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():

    for inputs, labels in test_loader:

        outputs = model(inputs)

        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

print(len(all_predictions))
print(len(all_labels))

In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for inputs, labels in test_loader:

        outputs = model(inputs)

        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

In [ ]:
print(len(all_predictions))
print(len(all_labels))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(12,10))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:

import pandas as pd
import os

In [ ]:
!pip install -q openai-whisper

In [ ]:
import whisper

print("Whisper installed successfully")

In [ ]:
whisper_model = whisper.load_model("base")

In [ ]:
sample_file = "/content/drive/MyDrive/TESS Toronto emotional speech set data/OAF_angry/OAF_germ_angry.wav"

result = whisper_model.transcribe(sample_file)

print(result["text"])

In [ ]:
texts = []
labels = []

In [ ]:
import os

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
dataset_path = "/content/TESS Toronto emotional speech set data"

In [ ]:
texts = []
labels = []

import os

In [ ]:
import os

In [ ]:
texts = []
labels = []

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/TESS Toronto emotional speech set data"

texts = []
labels = []

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):

        print("Processing:", folder)

        for file in os.listdir(folder_path):

            try:

                # REMOVE .wav
                filename = file.replace(".wav", "")

                # SPLIT NAME
                parts = filename.split("_")

                # GET WORD
                word = parts[1]

                texts.append(word)

                labels.append(folder)

            except Exception as e:

                print("ERROR:", file)
                print(e)

print("\nDONE")

print(texts[:10])

print(labels[:10])

In [ ]:
import pandas as pd

df = pd.DataFrame({

    "text": texts,
    "label": labels
})

print(df.head())

In [ ]:
import pandas as pd

df = pd.DataFrame({

    "text": texts,
    "label": labels
})

print(df.head())

In [ ]:
!pip install transformers

In [ ]:
from transformers import BertTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
encoder = LabelEncoder()

labels_encoded = encoder.fit_transform(labels)

print(labels_encoded[:10])

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels_encoded,
    test_size=0.2,
    random_state=42
)

In [ ]:
tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

In [ ]:
sample_text = train_texts[0]

tokens = tokenizer(
    sample_text,
    padding='max_length',
    truncation=True,
    max_length=10,
    return_tensors="pt"
)

print(tokens)

In [ ]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=10
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=10
)

In [ ]:
import torch

In [ ]:
class EmotionDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {

            key: torch.tensor(val[idx])

            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [ ]:
train_dataset = EmotionDataset(
    train_encodings,
    train_labels
)

test_dataset = EmotionDataset(
    test_encodings,
    test_labels
)

In [ ]:
from transformers import BertForSequenceClassification

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(set(labels_encoded))
)

In [ ]:
from transformers import Trainer
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(

    output_dir='./results',

    num_train_epochs=3,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    warmup_steps=100,

    weight_decay=0.01,

    logging_dir='./logs',

    logging_steps=10
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
predictions = trainer.predict(test_dataset)

print(predictions)

In [ ]:
import numpy as np

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

print(predicted_labels[:10])

In [ ]:
from sklearn.metrics import accuracy_score

text_accuracy = accuracy_score(
    test_labels,
    predicted_labels
)

print(f"Text Accuracy: {text_accuracy*100:.2f}%")

In [ ]:
speech_accuracy = 65
text_accuracy = 7.4

In [ ]:
fusion_accuracy = (
    0.7 * speech_accuracy
    +
    0.3 * text_accuracy
)

print(
    f"Fusion Accuracy: "
    f"{fusion_accuracy:.2f}%"
)

In [ ]:
import pandas as pd

results_df = pd.DataFrame({

    "Model": [
        "Speech CNN",
        "Text BERT",
        "Fusion Model"
    ],

    "Accuracy": [
        speech_accuracy,
        text_accuracy,
        fusion_accuracy
    ]
})

print(results_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Accuracy"]
)

plt.ylabel("Accuracy (%)")

plt.title("Model Comparison")

plt.show()

In [ ]:
plt.savefig("model_comparison.png")

In [ ]:
speech_model = SimpleCNN(num_classes)

In [ ]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):

    def __init__(self, num_classes):

        super(SimpleCNN, self).__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.flatten = nn.Flatten()

        self.fc = nn.Sequential(

            nn.Linear(
                64 * 10 * 50,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, x):

        x = self.cnn(x)

        x = self.flatten(x)

        x = self.fc(x)

        return x

In [ ]:
num_classes = 14

In [ ]:
speech_model = SimpleCNN(num_classes)

In [ ]:
import nbformat
import os

print(os.listdir("/content"))

In [ ]:
import nbformat

notebook_path = "/content/drive/MyDrive/Colab Notebooks/Copy of speech_pipeline.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open(notebook_path, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Widget metadata removed successfully!")